In [1]:
# Cell 1: Imports & Config

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DB_CONFIG = {
    "host": "100.105.75.47",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

TARGET_SCHEMA = "d1_machine_log"
TARGET_TABLE  = "FCT1_machine_log"

TARGET_END_DAY = "20260222"

MSG_START  = "제품 지그 해제 완료"
MSG_END    = "제품 안착 완료 ON"
MSG_MANUAL = "Manual mode 전환"

GAP_EXCLUDE_SEC = 60.0  # 60초 이상 제외
N_COMPONENTS = 5        # GMM K=5
RANDOM_STATE = 42

In [3]:
# Cell 2: DB connect & fetch minimal rows (FIX: quoted identifiers)

def make_engine(cfg: dict):
    url = (
        f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
    )
    return create_engine(url, pool_pre_ping=True)

engine = make_engine(DB_CONFIG)

# ✅ Postgres: 대문자/특수문자 포함 식별자는 반드시 " "로 감싸야 함
schema_q = f'"{TARGET_SCHEMA}"'
table_q  = f'"{TARGET_TABLE}"'          # TARGET_TABLE = "FCT1_machine_log"
full_q   = f"{schema_q}.{table_q}"

sql = text(f"""
    SELECT
        id,
        end_day,
        end_time,
        contents
    FROM {full_q}
    WHERE end_day = :end_day
      AND contents IN (:msg_start, :msg_end, :msg_manual)
    ORDER BY id ASC
""")

with engine.connect() as conn:
    df_raw = pd.read_sql(
        sql,
        conn,
        params={
            "end_day": TARGET_END_DAY,
            "msg_start": MSG_START,
            "msg_end": MSG_END,
            "msg_manual": MSG_MANUAL,
        },
    )

print(df_raw.shape)
df_raw.head(10)

(2874, 4)


,id,end_day,end_time,contents
0,1663193,20260222,00:00:04.54,제품 안착 완료 ON
1,1663200,20260222,00:00:35.69,제품 지그 해제 완료
2,1663203,20260222,00:00:40.62,제품 안착 완료 ON
3,1663210,20260222,00:01:14.49,제품 지그 해제 완료
4,1663213,20260222,00:01:19.46,제품 안착 완료 ON
5,1663220,20260222,00:01:50.63,제품 지그 해제 완료
6,1663223,20260222,00:01:55.69,제품 안착 완료 ON
7,1663230,20260222,00:02:28.51,제품 지그 해제 완료
8,1663233,20260222,00:02:33.41,제품 안착 완료 ON
9,1663240,20260222,00:03:06.03,제품 지그 해제 완료


In [4]:
# Cell 3: Parse timestamp

df = df_raw.copy()

# end_time 예: "08:56:42.13" (소수 2자리) → pandas가 알아서 %f 처리 가능(13 -> 130000으로 해석)
df["ts_str"] = df["end_day"].astype(str) + " " + df["end_time"].astype(str)

# 두 가지 포맷을 순차 시도(마이크로초 없는 케이스 대비)
ts = pd.to_datetime(df["ts_str"], format="%Y%m%d %H:%M:%S.%f", errors="coerce")
ts2 = pd.to_datetime(df["ts_str"], format="%Y%m%d %H:%M:%S", errors="coerce")

df["ts"] = ts.fillna(ts2)

bad = df["ts"].isna().sum()
if bad > 0:
    print(f"[WARN] timestamp parse failed rows: {bad}")
    display(df[df["ts"].isna()].head(20))

df = df.dropna(subset=["ts"]).sort_values(["ts", "id"], ascending=[True, True]).reset_index(drop=True)

df.head(10)

,id,end_day,end_time,contents,ts_str,ts
0,1663193,20260222,00:00:04.54,제품 안착 완료 ON,20260222 00:00:04.54,2026-02-22 00:00:04.540
1,1663200,20260222,00:00:35.69,제품 지그 해제 완료,20260222 00:00:35.69,2026-02-22 00:00:35.690
2,1663203,20260222,00:00:40.62,제품 안착 완료 ON,20260222 00:00:40.62,2026-02-22 00:00:40.620
3,1663210,20260222,00:01:14.49,제품 지그 해제 완료,20260222 00:01:14.49,2026-02-22 00:01:14.490
4,1663213,20260222,00:01:19.46,제품 안착 완료 ON,20260222 00:01:19.46,2026-02-22 00:01:19.460
5,1663220,20260222,00:01:50.63,제품 지그 해제 완료,20260222 00:01:50.63,2026-02-22 00:01:50.630
6,1663223,20260222,00:01:55.69,제품 안착 완료 ON,20260222 00:01:55.69,2026-02-22 00:01:55.690
7,1663230,20260222,00:02:28.51,제품 지그 해제 완료,20260222 00:02:28.51,2026-02-22 00:02:28.510
8,1663233,20260222,00:02:33.41,제품 안착 완료 ON,20260222 00:02:33.41,2026-02-22 00:02:33.410
9,1663240,20260222,00:03:06.03,제품 지그 해제 완료,20260222 00:03:06.03,2026-02-22 00:03:06.030


In [5]:
# Cell 4: Extract cycles and build DF1

def extract_cycles(df_logs: pd.DataFrame) -> pd.DataFrame:
    """
    Returns cycles with start_ts, end_ts, end_day, gap_time
    Rules:
      - Cycle = START -> END (strict order)
      - If MANUAL occurs between START and END -> exclude that cycle
      - If START occurs again while waiting for END:
          * discard previous START
          * new START begins a new candidate cycle (invalid resets)
      - END without START: ignore
    """
    cycles = []

    state = "waiting_start"
    start_ts = None
    start_day = None
    invalid = False

    for row in df_logs.itertuples(index=False):
        c = row.contents
        ts = row.ts
        day = row.end_day

        if state == "waiting_start":
            if c == MSG_START:
                state = "waiting_end"
                start_ts = ts
                start_day = day
                invalid = False
            else:
                # END or MANUAL without START => ignore
                continue

        elif state == "waiting_end":
            if c == MSG_MANUAL:
                invalid = True

            elif c == MSG_START:
                # 새 start가 들어오면 이전 start는 폐기, 새 start로 재시작 (manual 영향도 리셋)
                start_ts = ts
                start_day = day
                invalid = False

            elif c == MSG_END:
                if not invalid and start_ts is not None:
                    gap = (ts - start_ts).total_seconds()
                    cycles.append(
                        {
                            "end_day": start_day,   # start가 속한 end_day (현재는 동일 day만 조회했지만 안전하게)
                            "start_ts": start_ts,
                            "end_ts": ts,
                            "gap_time": gap,
                        }
                    )
                # END를 만나면 사이클 후보 종료 → 다음 start 대기
                state = "waiting_start"
                start_ts = None
                start_day = None
                invalid = False

    return pd.DataFrame(cycles)

df_cycles = extract_cycles(df)

print(df_cycles.shape)
df_cycles.head(20)

(1431, 4)


,end_day,start_ts,end_ts,gap_time
0,20260222,2026-02-22 00:00:35.690,2026-02-22 00:00:40.620,4.93
1,20260222,2026-02-22 00:01:14.490,2026-02-22 00:01:19.460,4.97
2,20260222,2026-02-22 00:01:50.630,2026-02-22 00:01:55.690,5.06
3,20260222,2026-02-22 00:02:28.510,2026-02-22 00:02:33.410,4.90
4,20260222,2026-02-22 00:03:06.030,2026-02-22 00:03:10.900,4.87
5,20260222,2026-02-22 00:03:43.630,2026-02-22 00:03:48.760,5.13
6,20260222,2026-02-22 00:04:21.730,2026-02-22 00:05:25.350,63.62
7,20260222,2026-02-22 00:05:56.490,2026-02-22 00:06:02.110,5.62
8,20260222,2026-02-22 00:06:35.130,2026-02-22 00:06:39.940,4.81
9,20260222,2026-02-22 00:07:30.130,2026-02-22 00:07:35.060,4.93


In [6]:
# Cell 5: Filter & build DF1

df1 = df_cycles.copy()

# 기본 sanity 필터: 0초 이하 제외
df1 = df1[df1["gap_time"] > 0].copy()

# 추가 조건: 60초 이상 제외
df1 = df1[df1["gap_time"] < GAP_EXCLUDE_SEC].copy()

# bigserial처럼 id 재생성
df1 = df1.sort_values(["start_ts", "end_ts"]).reset_index(drop=True)
df1.insert(0, "id", np.arange(1, len(df1) + 1, dtype=np.int64))

# 요구사항 컬럼만 남긴 DF1 (디버깅용으로 원하면 start_ts/end_ts를 남겨도 됨)
df1_req = df1[["id", "end_day", "gap_time"]].copy()

print(df1_req.shape)
df1_req.head(30)

(1396, 3)


,id,end_day,gap_time
0,1,20260222,4.93
1,2,20260222,4.97
2,3,20260222,5.06
3,4,20260222,4.90
4,5,20260222,4.87
5,6,20260222,5.13
6,7,20260222,5.62
7,8,20260222,4.81
8,9,20260222,4.93
9,10,20260222,5.02


In [7]:
# Cell 6: Fit GMM (K=5) and label ordering

if len(df1_req) < N_COMPONENTS:
    raise ValueError(f"DF1 rows({len(df1_req)}) < n_components({N_COMPONENTS}). 데이터가 너무 적습니다.")

X = df1_req[["gap_time"]].to_numpy(dtype=float)

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

gmm = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type="full",
    random_state=RANDOM_STATE,
    n_init=10,
    reg_covar=1e-6,
)
gmm.fit(Xz)

labels = gmm.predict(Xz)
df1_labeled = df1_req.copy()
df1_labeled["cluster_raw"] = labels

# cluster_raw별 gap_time 평균으로 정렬 → 1..5로 재매핑
cluster_means = (
    df1_labeled.groupby("cluster_raw")["gap_time"]
    .mean()
    .sort_values()
)

# raw_label -> ordered_label (1..5)
raw_to_ord = {raw: i+1 for i, raw in enumerate(cluster_means.index.tolist())}
df1_labeled["cluster"] = df1_labeled["cluster_raw"].map(raw_to_ord).astype(int)

name_map = {
    1: "min_outlier",
    2: "Q1",
    3: "median",
    4: "Q3",
    5: "max_outlier",
}
df1_labeled["cluster_name"] = df1_labeled["cluster"].map(name_map)

df1_labeled.head(30)

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\cluster\_kmea

,id,end_day,gap_time,cluster_raw,cluster,cluster_name
0,1,20260222,4.93,0,1,min_outlier
1,2,20260222,4.97,0,1,min_outlier
2,3,20260222,5.06,0,1,min_outlier
3,4,20260222,4.90,0,1,min_outlier
4,5,20260222,4.87,0,1,min_outlier
5,6,20260222,5.13,0,1,min_outlier
6,7,20260222,5.62,4,2,Q1
7,8,20260222,4.81,0,1,min_outlier
8,9,20260222,4.93,0,1,min_outlier
9,10,20260222,5.02,0,1,min_outlier


In [8]:
# Cell 7: Build DF2 (cluster summary)

df2 = (
    df1_labeled
    .groupby(["cluster", "cluster_name"], as_index=False)
    .agg(
        count=("gap_time", "size"),
        gap_min=("gap_time", "min"),
        gap_q1=("gap_time", lambda s: float(np.quantile(s, 0.25))),
        gap_median=("gap_time", "median"),
        gap_q3=("gap_time", lambda s: float(np.quantile(s, 0.75))),
        gap_max=("gap_time", "max"),
        gap_mean=("gap_time", "mean"),
        gap_std=("gap_time", "std"),
    )
    .sort_values("cluster")
    .reset_index(drop=True)
)

df2

,cluster,cluster_name,count,gap_min,gap_q1,gap_median,gap_q3,gap_max,gap_mean,gap_std
0,1,min_outlier,1080,4.66,4.890,4.970,5.040,5.30,4.967917,0.111572
1,2,Q1,167,5.32,5.765,6.290,6.915,7.96,6.388084,0.718844
2,3,median,98,8.05,9.425,10.530,11.875,15.33,10.751531,1.730124
3,4,Q3,26,15.64,18.970,23.455,28.975,35.01,23.965385,5.916895
4,5,max_outlier,25,37.99,44.690,47.170,50.420,59.48,47.629200,5.023149


In [9]:
# Cell 8 (Optional): Global quantiles reference (not GMM)

q = df1_req["gap_time"].quantile([0.0, 0.25, 0.5, 0.75, 1.0]).to_frame("gap_time")
q.rename(index={0.0:"min", 0.25:"Q1", 0.5:"median", 0.75:"Q3", 1.0:"max"}, inplace=True)
q

,gap_time
min,4.66
Q1,4.91
median,5.01
Q3,5.18
max,59.48
